In [1]:
%pip install -q pandas numpy pyarrow

import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

P = Path("/content/drive/MyDrive/Nasa App Challenges/PyroSync")
AGG, H6 = P / "data" / "aggregated" / "phase3", P / "data" / "harmonization" / "phase6"
OUT = P / "data" / "product" / "phase7"
OUT.mkdir(parents=True, exist_ok=True)

panel = pd.read_parquet(AGG / "panel_cell_month_2012_2021.parquet",
                        columns=["cell_id", "ix", "iy", "month", "viirs_all_n", "lat", "lon"])
hist = pd.read_parquet(H6 / "predicted_cell_month_pre2012.parquet")
st = pd.read_csv(H6 / "harmonized_monthly_state.csv", parse_dates=["month"]).sort_values("month").reset_index(drop=True)
rg = pd.read_csv(H6 / "harmonized_monthly_region.csv", parse_dates=["month"])
aoi = pd.read_csv(H6 / "aoi_error_by_block.csv")
meta = json.load(open(H6 / "phase6_metadata.json"))
for d in (panel, hist):
    d["month"] = pd.to_datetime(d["month"])

MONTHS = pd.date_range("2000-11-01", "2021-12-01", freq="MS")
REGIONS = ["North California", "Central California", "South California"]
assert len(MONTHS) == 254 and len(st) == 254 and (st["month"].to_numpy() == MONTHS.to_numpy()).all()
print("Loaded: panel", f"{len(panel):,}", "| pre-2012 cell rows", f"{len(hist):,}", "| months", len(MONTHS))

Mounted at /content/drive
Loaded: panel 538,713 | pre-2012 cell rows 16,535 | months 254


In [2]:
cells = (panel.drop_duplicates("cell_id")[["cell_id", "ix", "iy", "lat", "lon"]]
         .sort_values(["iy", "ix"]).reset_index(drop=True))
cells["region"] = np.select([cells["lat"] < 36, cells["lat"] < 38],
                            ["South California", "Central California"], "North California")
cells["cell_idx"] = cells.index
cidx = dict(zip(cells["cell_id"], cells["cell_idx"]))

obs = panel[panel["viirs_all_n"] > 0].assign(value=lambda d: d["viirs_all_n"].astype(float), src=0)
pre = hist[hist["pred"] > 0].assign(value=lambda d: d["pred"], src=lambda d: np.where(d["terra_only"], 2, 1))
cm = pd.concat([pre[["cell_id", "month", "value", "src"]], obs[["cell_id", "month", "value", "src"]]], ignore_index=True)
cm["cell_idx"] = cm["cell_id"].map(cidx)
cm["month_idx"] = MONTHS.get_indexer(cm["month"])
assert cm["cell_idx"].notna().all() and (cm["month_idx"] >= 0).all()
cm = cm.astype({"cell_idx": int, "month_idx": int, "src": int})
cm["value"] = cm["value"].round(3)
assert not cm.duplicated(["cell_idx", "month_idx"]).any()

tot = cm.groupby("month_idx")["value"].sum().reindex(range(254), fill_value=0).to_numpy()
assert np.allclose(tot, st["value"].to_numpy(), atol=0.5), "cell table does not add up to the Phase 6 statewide series"
print("Cells:", len(cells), "| non-zero cell-months:", f"{len(cm):,}")
print(cm["src"].map({0: "observed", 1: "predicted (MODIS)", 2: "predicted (Terra-only)"}).value_counts().to_string())

Cells: 4527 | non-zero cell-months: 48,059
src
observed                  31545
predicted (MODIS)         14773
predicted (Terra-only)     1741


In [3]:
allser = pd.concat([rg, st], ignore_index=True)
allser["year"], allser["m"] = allser["month"].dt.year, allser["month"].dt.month
cal = allser.pivot(index=["region", "year"], columns="m", values="value")
clim = (allser.groupby(["region", "m"])["value"]
        .agg(median="median", q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75),
             min="min", max="max", n_years="size").reset_index())

last_m = MONTHS[-1]
rows = []
for r in ["California"] + REGIONS:
    s = allser[(allser["region"] == r) & (allser["m"] == last_m.month)]
    v = float(s.loc[s["month"] == last_m, "value"].iloc[0]); med = float(clim[(clim.region == r) & (clim.m == last_m.month)]["median"].iloc[0])
    rows.append({"region": r, "month": last_m.strftime("%Y-%m"), "value": v, "median_same_month": med,
                 "vs_median": v / med if med > 0 else np.nan, "percentile_rank": float((s["value"] < v).mean())})
latest = pd.DataFrame(rows)
display(clim[clim["region"] == "California"].round(1))
display(latest.round(3))

,region,m,median,q25,q75,min,max,n_years
0,California,1,590.1,306.0,922.0,112.7,1286.2,21
1,California,2,765.8,543.7,1010.6,245.3,2237.0,21
2,California,3,899.0,631.8,1317.6,234.9,1990.6,21
3,California,4,887.0,652.0,1046.0,311.1,1501.7,21
4,California,5,777.8,577.8,884.0,395.0,1541.0,21
5,California,6,1338.0,926.0,2558.0,553.9,24429.8,21
6,California,7,2958.1,2071.0,6964.0,1000.0,41165.3,21
7,California,8,11321.0,2932.4,21923.0,1231.1,105152.0,21
8,California,9,6182.0,3728.1,9569.0,1620.1,130744.0,21
9,California,10,3652.0,1905.0,5196.1,849.0,25841.0,21


,region,month,value,median_same_month,vs_median,percentile_rank
0,California,2021-12,876.0,711.392,1.231,0.636
1,North California,2021-12,489.0,330.719,1.479,0.727
2,Central California,2021-12,205.0,192.000,1.068,0.545
3,South California,2021-12,182.0,145.000,1.255,0.591


In [4]:
TOL = 0.30
n_reg = int(len(cells) / 3)
curve = list(zip(aoi["block_cells"], aoi["WAPE_month"])) + [(n_reg, meta["cv_wape_selected"])]
CURVE_N = np.array([c for c, _ in curve], float)
CURVE_E = np.minimum.accumulate(np.array([e for _, e in curve], float))   # non-increasing with area

def min_km(col):
    ok = aoi[aoi[col] <= TOL]
    return int(ok["block_km"].min()) if len(ok) else None

cfg = {"error_tolerance": TOL, "min_block_km_monthly": min_km("WAPE_month"), "min_block_km_yearly": min_km("WAPE_year"),
       "error_curve_cells": CURVE_N.tolist(), "error_curve_wape": CURVE_E.tolist()}
print(pd.DataFrame({"cells": CURVE_N, "typical_monthly_error": CURVE_E}).round(3).to_string(index=False))
print("Minimum area for monthly values:", cfg["min_block_km_monthly"], "km | for yearly values:", cfg["min_block_km_yearly"], "km")

 cells  typical_monthly_error
   1.0                  0.339
   4.0                  0.291
  16.0                  0.242
  64.0                  0.205
 256.0                  0.176
1509.0                  0.129
Minimum area for monthly values: 20 km | for yearly values: 10 km


In [5]:
src_by_month = st["source"].to_numpy()
min_cells = (cfg["min_block_km_monthly"] / 10) ** 2 if cfg["min_block_km_monthly"] else np.inf

def aoi_series(cell_ids):
    ci = cells.loc[cells["cell_id"].isin(cell_ids), "cell_idx"].to_numpy()
    n = len(ci); assert n > 0, "empty AOI"
    v = (cm[cm["cell_idx"].isin(ci)].groupby("month_idx")["value"].sum()
         .reindex(range(254), fill_value=0).to_numpy())
    err = float(np.interp(np.log(n), np.log(CURVE_N), CURVE_E))
    pred = src_by_month != "observed VIIRS"
    return pd.DataFrame({"month": MONTHS.strftime("%Y-%m"), "value": v.round(2),
                         "lo": np.where(pred, v * (1 - err), v).round(2), "hi": np.where(pred, v * (1 + err), v).round(2),
                         "source": src_by_month, "n_cells": n,
                         "reliable": (~pred) | (n >= min_cells)})

for r in REGIONS:
    a = aoi_series(cells.loc[cells["region"] == r, "cell_id"])
    ref = rg[rg["region"] == r].sort_values("month")["value"].to_numpy()
    assert np.allclose(a["value"], ref, atol=0.5), r
assert np.allclose(aoi_series(cells["cell_id"])["value"], st["value"], atol=0.5)
print("AOI function reproduces all three region series and the statewide series.")
one = aoi_series(cells.loc[cells["region"] == "Central California", "cell_id"].iloc[:1])
print("Single-cell AOI: reliable months", int(one["reliable"].sum()), "of", len(one), "| predicted months flagged:", int((~one["reliable"]).sum()))

AOI function reproduces all three region series and the statewide series.
Single-cell AOI: reliable months 119 of 254 | predicted months flagged: 135


In [6]:
cells[["cell_idx", "cell_id", "ix", "iy", "lat", "lon", "region"]].to_csv(OUT / "cells.csv", index=False)
cm[["cell_idx", "month_idx", "value", "src"]].sort_values(["month_idx", "cell_idx"]).to_csv(OUT / "cell_month_sparse.csv.gz", index=False)
pd.DataFrame({"month_idx": range(254), "month": MONTHS.strftime("%Y-%m"), "source": src_by_month}).to_csv(OUT / "months.csv", index=False)
allser[["month", "region", "value", "lo", "hi", "source"]].to_csv(OUT / "series_long.csv", index=False)
cal.to_csv(OUT / "calendar_wide.csv")
clim.to_csv(OUT / "climatology.csv", index=False)
cfg.update({"sources": {"0": "observed VIIRS", "1": "predicted (MODIS to VIIRS)", "2": "predicted (Terra-only)"},
            "data_start": "2000-11", "data_end": "2021-12", "first_observed_month": "2012-02",
            "latest_vs_median": latest.to_dict("records"),
            "notes": ["Values are VIIRS-equivalent active-fire detection counts, not fires or burned area.",
                      "Months before 2012-02 are predicted from MODIS and cannot be validated against VIIRS.",
                      "Predicted values are only reliable when summed over areas at least the minimum size."]})
with open(OUT / "app_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

sizes = {f.name: round(f.stat().st_size / 1e6, 2) for f in OUT.iterdir()}
print(pd.Series(sizes).to_string())
assert sum(sizes.values()) < 5, "product larger than expected for a static site"
print("\nPHASE 7 COMPLETED:", len(sizes), "files in", OUT)

cells.csv                   0.36
cell_month_sparse.csv.gz    0.17
months.csv                  0.01
series_long.csv             0.09
calendar_wide.csv           0.01
climatology.csv             0.00
app_config.json             0.00

PHASE 7 COMPLETED: 7 files in /content/drive/MyDrive/Nasa App Challenges/PyroSync/data/product/phase7
